### Parsing CSV Files
 

In [1]:
import pandas as pd
import os

In [2]:
os.makedirs("data/structured_files", exist_ok=True)

In [3]:
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics'],
    'Price': [999.99, 29.99, 79.99, 299.99, 89.99],
    'Stock': [50, 200, 150, 75, 100],
    'Description': [
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wireless optical mouse with ergonomic design',
        'Mechanical keyboard with RGB backlighting',
        '27-inch 4K monitor with HDR support',
        '1080p webcam with noise cancellation'
    ]
}

df = pd.DataFrame(data)
df.to_csv("data/structured_files/products.csv", index=False)


In [6]:
with pd.ExcelWriter("data/structured_files/inventory.xlsx")  as writer:
    df.to_excel(writer, sheet_name="Products", index=False)

    summary_data = {
    'Category': ['Electronics', 'Accessories'],
    'Total_Items': [3, 2],
    'Total_Value': [1389.97, 109.98]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name="Summary", index=False)

    

In [10]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

csv_loader = CSVLoader(file_path="data/structured_files/products.csv", encoding = "utf-8",csv_args={
    "delimiter": ",",
    "quotechar": '"',
})
csv_docs = csv_loader.load()
print(f"Length of docs {len(csv_docs)}")
for i, doc in enumerate(csv_docs):
    print(f"Doc Content: {doc.page_content}")
    print(f"Doc Metadata: {doc.metadata}")





Length of docs 5
Doc Content: Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD
Doc Metadata: {'source': 'data/structured_files/products.csv', 'row': 0}
Doc Content: Product: Mouse
Category: Accessories
Price: 29.99
Stock: 200
Description: Wireless optical mouse with ergonomic design
Doc Metadata: {'source': 'data/structured_files/products.csv', 'row': 1}
Doc Content: Product: Keyboard
Category: Accessories
Price: 79.99
Stock: 150
Description: Mechanical keyboard with RGB backlighting
Doc Metadata: {'source': 'data/structured_files/products.csv', 'row': 2}
Doc Content: Product: Monitor
Category: Electronics
Price: 299.99
Stock: 75
Description: 27-inch 4K monitor with HDR support
Doc Metadata: {'source': 'data/structured_files/products.csv', 'row': 3}
Doc Content: Product: Webcam
Category: Electronics
Price: 89.99
Stock: 100
Description: 1080p webcam with noise cancellation
Doc Metadata: {'source': 'data/struct

In [14]:
from langchain_community.document_loaders import UnstructuredExcelLoader

excel_loader = UnstructuredExcelLoader(file_path="data/structured_files/inventory.xlsx", mode="elements")
unstructued_excel = excel_loader.load()
 
print(unstructued_excel) 
print(f"Length of excel {len(unstructued_excel)}")
for index, doc in enumerate(unstructued_excel):
    print(f"document content {index+1}: {doc.page_content}")
    print(f"Metdata {index+1}: {doc.metadata}")

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'file_directory': 'data/structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-05-18T17:23:44', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>50</td><td>High-performance laptop with 16GB RAM and 512GB SSD</td></tr><tr><td>Mouse</td><td>Accessories</td><td>29.99</td><td>200</td><td>Wireless optical mouse with ergonomic design</td></tr><tr><td>Keyboard</td><td>Accessories</td><td>79.99</td><td>150</td><td>Mechanical keyboard with RGB backlighting</td></tr><tr><td>Monitor</td><td>Electronics</td><td>299.99</td><td>75</td><td>27-inch 4K monitor with HDR support</td></tr><tr><td>Webcam</td><td>Electronics</td><td>89.99</td><td>100</td><td>1080p webcam with noise cancellation</td></tr></table>', 'languages': ['eng'], 'filetype': 'applicatio

In [27]:
print("\n2️⃣ Custom CSV Processing")
from typing import List
from langchain_core.documents import Document

def process_csv_intelligently(path) -> List[Document]:
    df = pd.read_csv(path)
    documents = []
    for idx, row in df.iterrows():
        print(row)
        content = f"""Product Information:
        Name: {row['Product']}
        Category: {row['Category']}
        Price: {row['Price']}
        Stock: {row['Stock']} units
        Description: {row['Description']}"""
        doc = Document(
            page_content= content,
            metadata={
                "source": path,
                "row_index": idx,
                "product_name": row['Product'],
                "category": row['Category'],
                "price": row['Price'],
                "data_type": 'product_info'
            }
        )
        documents.append(doc)
    return documents    




2️⃣ Custom CSV Processing


In [35]:
def process_excel_intelligently(path):
    documents = []
    excel_file = pd.ExcelFile(path)
    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet_name)
        print(df.columns)
        sheet_content = f"Sheet: {sheet_name}"
        sheet_content += f"Columns {df.columns}"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)

        doc = Document(
            page_content=sheet_content,
            metadata={
                "source": path,
                "sheet_name": sheet_name,
                "num_rows": len(df),
                "num_columns": len(df.columns),
                "data_type": "excel_sheet"
            }
        )
        documents.append(doc)
    return documents    

process_excel_intelligently("data/structured_files/inventory.xlsx")

Index(['Product', 'Category', 'Price', 'Stock', 'Description'], dtype='str')
Index(['Category', 'Total_Items', 'Total_Value'], dtype='str')


[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content="Sheet: ProductsColumns Index(['Product', 'Category', 'Price', 'Stock', 'Description'], dtype='str')Rows: 5\n\n Product    Category  Price  Stock                                         Description\n  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD\n   Mouse Accessories  29.99    200        Wireless optical mouse with ergonomic design\nKeyboard Accessories  79.99    150           Mechanical keyboard with RGB backlighting\n Monitor Electronics 299.99     75                 27-inch 4K monitor with HDR support\n  Webcam Electronics  89.99    100                1080p webcam with noise cancellation"),
 Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type': 'excel_sheet'}, page_content="Sheet: SummaryCol

In [28]:
process_csv_intelligently("data/structured_files/products.csv")

Product                                                   Laptop
Category                                             Electronics
Price                                                     999.99
Stock                                                         50
Description    High-performance laptop with 16GB RAM and 512G...
Name: 0, dtype: object
Product                                               Mouse
Category                                        Accessories
Price                                                 29.99
Stock                                                   200
Description    Wireless optical mouse with ergonomic design
Name: 1, dtype: object
Product                                         Keyboard
Category                                     Accessories
Price                                              79.99
Stock                                                150
Description    Mechanical keyboard with RGB backlighting
Name: 2, dtype: object
Product              

[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'data_type': 'product_info'}, page_content='Product Information:\n        Name: Laptop\n        Category: Electronics\n        Price: 999.99\n        Stock: 50 units\n        Description: High-performance laptop with 16GB RAM and 512GB SSD'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product_name': 'Mouse', 'category': 'Accessories', 'price': 29.99, 'data_type': 'product_info'}, page_content='Product Information:\n        Name: Mouse\n        Category: Accessories\n        Price: 29.99\n        Stock: 200 units\n        Description: Wireless optical mouse with ergonomic design'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 2, 'product_name': 'Keyboard', 'category': 'Accessories', 'price': 79.99, 'data_type': 'product_info'}, page_content='Product Informati